In [ ]:
%run migraine_clustering_pipeline.ipynb

In [ ]:
from itertools import permutations
from sklearn.metrics import adjusted_rand_score

LATENT_DIMS = (10, 20, 30)
PAIRS = ((10, 20), (20, 30))


def consistent_assignment_pct(labels_a, labels_b):
    """Percentage of subjects with the same cluster after the best matching of labels."""
    labels_a, labels_b = np.asarray(labels_a), np.asarray(labels_b)
    clusters = np.unique(np.concatenate([labels_a, labels_b]))
    best = 0
    for perm in permutations(clusters):
        mapping = dict(zip(clusters, perm))
        mapped = np.array([mapping[l] for l in labels_b])
        best = max(best, np.mean(labels_a == mapped))
    return 100 * best

In [ ]:
labels_by_dim = {}
for d in LATENT_DIMS:
    res_d, _ = run_clustering_pipeline(df_features, k_best=K_BEST, latent_dim=d)
    labels_by_dim[d] = {m: r["labels"] for m, r in res_d.items()}

rows = []
for method in ("kmeans", "agglomerative", "spectral"):
    for d1, d2 in PAIRS:
        l1, l2 = labels_by_dim[d1][method], labels_by_dim[d2][method]
        rows.append({"group": GROUP, "model": MODEL, "method": method,
                     "comparison": f"{d1} vs {d2}",
                     "ARI": round(adjusted_rand_score(l1, l2), 3),
                     "consistent_%": round(consistent_assignment_pct(l1, l2), 1)})

sens_df = pd.DataFrame(rows)
print(sens_df.to_string(index=False))
os.makedirs("results", exist_ok=True)
sens_df.to_csv(f"results/latent_sensitivity_{GROUP}_{MODEL}.csv", index=False)